# Створення додатків для генерації зображень (OpenAI)

LLM — це не лише генерація тексту. Ви також можете створювати зображення на основі текстових описів. Зображення як модальність корисні у медичних технологіях, архітектурі, туризмі, розробці ігор, маркетингу та інших сферах. У цьому уроку ми працюємо з сучасними моделями **GPT Image** на платформі OpenAI.

## Цілі навчання

До кінця цього уроку ви зможете:

- Пояснити, що таке генерація зображень і де вона застосовується.
- Зрозуміти сімейство моделей `gpt-image` і чим вони відрізняються від застарілих моделей DALL·E.
- Створити додаток для генерації зображень і редагувати зображення.

## Що таке генерація зображень?

Моделі генерації зображень створюють зображення за текстовим запитом. Сучасні моделі, такі як `gpt-image`, під час навчання вивчають зв’язок між текстом і зображеннями, а потім ітеративно перетворюють випадковий шум у зображення, що відповідає вашому запиту.

Дві відомі сімейства моделей зображень:

- **`gpt-image` (OpenAI)** — поточне покоління, що використовується у цьому уроці. Підтримує генерацію текст-до-зображення та редагування зображень (інпейнтинг за маскою).
- **Midjourney** — популярна модель стороннього розробника з власним сервісом і робочим процесом на базі Discord.

> Старі моделі зображень OpenAI — **DALL·E 2** і **DALL·E 3** — застарілі. DALL·E 3 більше недоступний для нових розгортань, а такі можливості, як `create_variation`, були лише в DALL·E 2. Використовуйте моделі `gpt-image` для нових додатків.

> **Важливо:** моделі `gpt-image` повертають згенероване зображення у вигляді **base64** (`b64_json`), а не URL. Ваш код декодує рядок base64 у байти і зберігає його — URL зображення для завантаження відсутній.


## Створення вашого першого додатку для генерації зображень

Отже, що потрібно для створення додатку для генерації зображень? Вам знадобляться такі бібліотеки:

- **python-dotenv**, настійно рекомендується використовувати цю бібліотеку, щоб зберігати ваші секрети у файлі *.env* подалі від коду.
- **openai**, ця бібліотека потрібна для взаємодії з API OpenAI.
- **pillow**, щоб працювати з зображеннями у Python.


1. Створіть файл *.env* з наступним вмістом:

    ```text
    OPENAI_API_KEY='<add your OpenAI key here>'
    ```



1. Зберіть вищезазначені бібліотеки у файл під назвою *requirements.txt* ось так:

    ```text
    python-dotenv
    openai
    pillow
    ```


1. Далі створіть віртуальне середовище та встановіть бібліотеки:


In [ ]:
# create virtual env
! python3 -m venv venv
# activate environment
! source venv/bin/activate
# install libraries
# pip install -r requirements.txt, if using a requirements.txt file 
! pip install python-dotenv openai pillow

> [!NOTE]
> Для Windows використовуйте наступні команди, щоб створити та активувати ваше віртуальне середовище:

    ```bash
    python3 -m venv venv
    venv\Scripts\activate.bat
    ````

1. Додайте наступний код у файл під назвою *app.py*:

    ```python
    import os
    import base64
    from PIL import Image
    import dotenv
    import openai

    # import dotenv
    dotenv.load_dotenv()

    # Create OpenAI object (reads OPENAI_API_KEY from your .env)
    client = openai.OpenAI()


    try:
        # Create an image by using the image generation API
        generation_response = client.images.generate(
            model="gpt-image-2",
            prompt='Bunny on horse, holding a lollipop, on a foggy meadow where it grows daffodils',    # Enter your prompt text here
            size='1024x1024',
            n=1
        )
        # Set the directory for the stored image
        image_dir = os.path.join(os.curdir, 'images')

        # If the directory doesn't exist, create it
        if not os.path.isdir(image_dir):
            os.mkdir(image_dir)

        # Initialize the image path (note the filetype should be png)
        image_path = os.path.join(image_dir, 'generated-image.png')

        # gpt-image models return the image as base64 (b64_json), not a URL
        image_b64 = generation_response.data[0].b64_json
        generated_image = base64.b64decode(image_b64)
        with open(image_path, "wb") as image_file:
            image_file.write(generated_image)

        # Display the image in the default image viewer
        image = Image.open(image_path)
        image.show()

    # catch exceptions
    except openai.BadRequestError as err:
        print(err)

    ```

Пояснимо цей код:

- Спершу ми імпортуємо потрібні бібліотеки, зокрема бібліотеку OpenAI, бібліотеку dotenv, модуль base64 та бібліотеку Pillow.

    ```python
    import os
    import base64
    from PIL import Image
    import dotenv
    import openai
    ```

- Після цього створюємо клієнта, що зчитує API ключ з вашого ``.env``.

    ```python
    # Create OpenAI object
    client = openai.OpenAI()
    ```

- Далі генеруємо зображення:

    ```python
    generation_response = client.images.generate(
        model="gpt-image-2",
        prompt='Bunny on horse, holding a lollipop, on a foggy meadow where it grows daffodils',    # Enter your prompt text here
        size='1024x1024',
        n=1
    )
    ```

    `gpt-image` моделі повертають зображення у вигляді **base64** рядка в `data[0].b64_json`. Ми декодуємо його у байти та записуємо у файл — немає URL для завантаження.

    ```python
    image_b64 = generation_response.data[0].b64_json
    generated_image = base64.b64decode(image_b64)
    with open(image_path, "wb") as image_file:
        image_file.write(generated_image)
    ```

- Нарешті, відкриваємо зображення та використовуємо стандартний переглядач зображень для його показу:

    ```python
    image = Image.open(image_path)
    image.show()
    ```

### Детальніше про генерацію зображень

Розглянемо параметри `images.generate`:

- **model** — модель зображення, яку слід використовувати (наприклад, `gpt-image-2`).
- **prompt** — текстовий запит, що використовується для генерації зображення. Тут це "Bunny on horse, holding a lollipop, on a foggy meadow where it grows daffodils".
- **size** — розмір згенерованого зображення (наприклад, `1024x1024`, `1536x1024`, `1024x1536` або `"auto"`).
- **n** — кількість зображень для генерації. Тут ми генеруємо одне.

> Моделі зображень не приймають параметр `temperature` — це керування для генерації тексту. Щоб отримати різноманітність, викликайте API знову; щоб зменшити різноманітність, зробіть ваш запит більш конкретним.

## Додаткові можливості генерації зображень

Ви побачили, як генерувати зображення за допомогою кількох рядків Python. `gpt-image` моделі також можуть **редагувати** існуюче зображення. Надані зображення, необов'язкова **маска** (яка позначає область для зміни) та запит дозволяють змінити частину зображення — наприклад, додати капелюх нашому кролику.

```python
response = client.images.edit(
  model="gpt-image-2",
  image=open("base_image.png", "rb"),
  mask=open("mask.png", "rb"),
  prompt="An image of a rabbit with a hat on its head.",
  n=1,
  size="1024x1024"
)
# edits are also returned as base64
edited_image = base64.b64decode(response.data[0].b64_json)
```

Початкове зображення містить лише кролика; кінцеве зображення має капелюх на кролику.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Відмова від відповідальності**:
Цей документ було перекладено за допомогою сервісу штучного інтелекту для перекладу [Co-op Translator](https://github.com/Azure/co-op-translator). Хоча ми прагнемо до точності, будь ласка, майте на увазі, що автоматичні переклади можуть містити помилки або неточності. Оригінальний документ рідною мовою слід вважати авторитетним джерелом. Для критично важливої інформації рекомендується професійний людський переклад. Ми не несемо відповідальності за будь-які непорозуміння або неправильні тлумачення, що виникли внаслідок використання цього перекладу.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
